In [0]:
import pandas as pd
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
# File path in DBFS
file_path = "/Volumes/workspace/default/amazon_review_raw/All_Beauty.jsonl"

# Read into Spark DataFrame
bronze_df = spark.read.json(file_path, multiLine=False)

# Show schema
bronze_df.printSchema()

# Quick preview
bronze_df.show(5, truncate=100)



root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

+----------+------------+------+-----------+------+----------------------------------------------------------------------------------------------------+-------------+-----------------------------------------+----------------------------+-----------------+
|      asin|helpful_vote|images|parent_asin|rating|             

In [0]:
# Select and standardize bronze schema
# Get last item of filepath

bronze_df = bronze_df.select(
    col("user_id"),
    col("rating"),
    col("asin").alias("product_id"),
    col("text").alias("review_text"),
    col("timestamp").alias("review_timestamp")
).withColumn("category", lit("All_Beauty")) \
 .withColumn("ingestion_ts", current_timestamp()) \
 .withColumn("source", lit("All_Beauty.jsonl"))

In [0]:
bronze_df.show()

+--------------------+------+----------+--------------------+----------------+----------+--------------------+----------------+
|             user_id|rating|product_id|         review_text|review_timestamp|  category|        ingestion_ts|          source|
+--------------------+------+----------+--------------------+----------------+----------+--------------------+----------------+
|AGKHLEW2SOWHNMFQI...|   5.0|B00YQ6X8EO|This spray is rea...|   1588687728923|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|
|AGKHLEW2SOWHNMFQI...|   4.0|B081TJ8YS3|This product does...|   1588615855070|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|
|AE74DYR3QUGVPZJ3P...|   5.0|B07PNNCSP9|Smells good, feel...|   1589665266052|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|
|AFQLNQNQYFWQZPJQZ...|   1.0|B09JS339BZ|      Felt synthetic|   1643393630220|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|
|AFQLNQNQYFWQZPJQZ...|   5.0|B08BZ63GMJ|             Love it|   1609322563534|All_Beauty|2026-03-01 18:0

In [0]:
# Write Bronze delta table
bronze_table_name = "bronze_amazon_all_beauty"
bronze_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(bronze_table_name)

# Verify
spark.sql(f"SELECT COUNT(*) FROM {bronze_table_name}").show()


+--------+
|COUNT(*)|
+--------+
|  701528|
+--------+



In [0]:
# Quick sanity check
spark.sql(f"""
SELECT
    category,
    COUNT(*) AS reviews,
    MIN(review_timestamp) AS min_date,
    MAX(review_timestamp) AS max_date
FROM {bronze_table_name}
GROUP BY category
""").show()


+----------+-------+------------+-------------+
|  category|reviews|    min_date|     max_date|
+----------+-------+------------+-------------+
|All_Beauty| 701528|973052658000|1694219976666|
+----------+-------+------------+-------------+

